# Stage-2 Gate 0.1 legacy-inventory recovery

Recovery-only, unexecuted operator for the completed v7 evidence. **Runtime → Run all** requires only Google Drive authorization. It performs an early metadata preflight, reuses and verifies the sealed A100/20/200-step evidence on CPU, imports P:0006 for development/model assessment only, seals evaluation readiness, and stops. It cannot launch 100k training, pilots, ablations, P:0007, or P:0009.

In [ ]:
from pathlib import Path
import os, re, subprocess

REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
TRAINING_EVIDENCE_COMMIT = '82633d66e5ea47f96b149ea22cc192fcf4526f06'
OPERATOR_IMPLEMENTATION_COMMIT = 'b1de7efcec4a81463db19777b15341f36f064f46'
if OPERATOR_IMPLEMENTATION_COMMIT == '__OPERATOR_IMPLEMENTATION_COMMIT__':
    raise RuntimeError('Notebook sealing commit has not pinned the operator implementation.')
if re.fullmatch(r'[0-9a-f]{40}', OPERATOR_IMPLEMENTATION_COMMIT) is None:
    raise ValueError('Operator implementation must be an exact 40-character Git SHA.')
REPO_DIR = Path('/content/MRIxFields-stage2-gate01-recovery-v8')
if REPO_DIR.exists():
    raise FileExistsError('Start a fresh runtime; refusing to mutate an existing checkout.')
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin', OPERATOR_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', OPERATOR_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
def git_text(*args):
    return subprocess.check_output(['git', *args], cwd=REPO_DIR, text=True).strip()
if git_text('rev-parse', 'HEAD') != OPERATOR_IMPLEMENTATION_COMMIT:
    raise RuntimeError('Detached operator implementation SHA mismatch.')
if subprocess.run(['git', 'merge-base', '--is-ancestor', TRAINING_EVIDENCE_COMMIT, 'HEAD'], cwd=REPO_DIR).returncode:
    raise RuntimeError('Operator commit does not descend from the training-evidence commit.')
changed_paths = git_text('diff', '--name-only', TRAINING_EVIDENCE_COMMIT, 'HEAD').splitlines()
allowed_exact = {
    'src/fieldbridge/evaluation/stage2_unified_gate01_p0006.py',
    'src/fieldbridge/evaluation/stage2_unified_preflight.py',
}
allowed_prefixes = ('notebooks/', 'tests/', 'docs/')
disallowed = [path for path in changed_paths if path not in allowed_exact and not path.startswith(allowed_prefixes)]
if disallowed:
    raise RuntimeError({'operator_diff_touches_training_critical_code': disallowed})
training_critical_prefixes = (
    'src/fieldbridge/models/', 'src/fieldbridge/training/', 'configs/',
    'src/fieldbridge/data/photometry_factored_latent_bank.py',
    'src/fieldbridge/data/photometry_factored_bank_dataset.py',
)
if any(path.startswith(training_critical_prefixes) for path in changed_paths):
    raise RuntimeError('Training, model, optimizer, bank, config, or loss code changed.')
if git_text('status', '--porcelain') or subprocess.run(['git', 'symbolic-ref', '-q', 'HEAD'], cwd=REPO_DIR, capture_output=True).returncode == 0:
    raise RuntimeError('Operator checkout must be detached and clean.')
CLI_ENV = os.environ.copy()
print({'training_evidence_commit': TRAINING_EVIDENCE_COMMIT, 'operator_implementation_commit': OPERATOR_IMPLEMENTATION_COMMIT, 'operator_descends_from_training_evidence': True, 'training_critical_code_byte_identical': True, 'detached_clean_checkout': True}, flush=True)
operator_path = REPO_DIR / 'notebooks/stage2_gate01_legacy_recovery_operator.py'
exec(compile(operator_path.read_text(encoding='utf-8'), str(operator_path), 'exec'), globals())
